# Popularity bias in recommenders

Cleans the raw Amazon Reviews 2023 .jsonl and splits it by time (`split_json.py`), then trains and evaluates
three recommenders (`mostpop`, `itemcf`, `mf`) on accuracy and popularity-bias metrics (`run.py`).

Run this notebook from the project folder. Data: https://amazon-reviews-2023.github.io/

## Settings

In [1]:
CATEGORY = "Clothing_Shoes_and_Jewelry"
RAW_REVIEWS = f"data/{CATEGORY}.jsonl.gz"
K_CORE = 2
MODELS = ["mostpop", "itemcf", "mf"]

## Download the raw reviews
Downloads `<CATEGORY>.jsonl.gz` from the Amazon Reviews 2023 site into `data/` (Clothing is ~7 GB).
Skipped if the raw file or the train/val/test split already exists.

In [ ]:
import os
import urllib.request

URL = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/{CATEGORY}.jsonl.gz"
SPLIT_DONE = os.path.exists(f"data/{CATEGORY}_k{K_CORE}_train.parquet")

def progress(n, size, total):
    if n % 12_000 == 0:  # about every 100 MB
        print(f"\r{n * size / 1e9:.1f} / {total / 1e9:.1f} GB", end="")

if SPLIT_DONE or os.path.exists(RAW_REVIEWS):
    print("Already have the data, skipping download")
else:
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(URL, RAW_REVIEWS + ".part", progress)  # .part so a failed download isn't mistaken for a full one
    os.replace(RAW_REVIEWS + ".part", RAW_REVIEWS)
    print("\nDone")

## Clean and split into train / validation / test
Runs `split_json.py` on the raw reviews: keeps verified purchases (latest review per user and item),
applies the k-core filter and makes `data/<CATEGORY>_k<K_CORE>_{train,val,test}.parquet`.
Skipped if the split already exists.

In [ ]:
import sys
PY = sys.executable  # run the scripts with this notebook's Python, not whatever "python" is on PATH

if SPLIT_DONE:
    print("Split already exists, skipping split_json.py")
else:
    !"{PY}" split_json.py {RAW_REVIEWS} {CATEGORY} {K_CORE}

## Train and evaluate

In [5]:
for m in MODELS:
    !"{PY}" run.py {CATEGORY} {K_CORE} {m}

Traceback (most recent call last):
  File "c:\Users\eekan\Downloads\Data cleaning\popularity-bias-recsys\run.py", line 28, in <module>
    train = pd.read_parquet(f"data/{cat}_k{k_core}_train.parquet")
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eekan\anaconda3\envs\ir\Lib\site-packages\pandas\io\parquet.py", line 760, in read_parquet
    return impl.read(
           ^^^^^^^^^^
  File "c:\Users\eekan\anaconda3\envs\ir\Lib\site-packages\pandas\io\parquet.py", line 296, in read
    pa_table = self.api.parquet.read_table(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eekan\anaconda3\envs\ir\Lib\site-packages\pyarrow\parquet\core.py", line 1915, in read_table
    dataset = ParquetDataset(
              ^^^^^^^^^^^^^^^
  File "c:\Users\eekan\anaconda3\envs\ir\Lib\site-packages\pyarrow\parquet\core.py", line 1491, in __init__
    self._dataset = ds.dataset(path_or_paths, filesystem=filesystem,
                    ^^^^^^^^^^^^^^^^^^^^^^^^